# 09 — Hospital Data Simulation Lab

Mục tiêu: sinh train x5/x10 từ Cleveland, mô phỏng dữ liệu bệnh viện bị thiếu/sai/outlier và đo độ bền model. Synthetic generator **chỉ fit trên train**. Test sạch không tham gia sinh dữ liệu. Đây là simulation phục vụ nghiên cứu, không thay thế dữ liệu bệnh viện thật hoặc external validation.

In [ ]:
!pip install -q gdown sdv xgboost lightgbm

In [ ]:
import time
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option('display.max_columns', 100)

## 1. Tải dữ liệu và giữ test thật

Test 20% được khóa trước mọi thao tác sinh dữ liệu. Trong lab này, test được dùng để khảo sát robustness; khi chốt model thật vẫn cần một external test từ bệnh viện khác.

In [ ]:
FILE_ID = '1YTzUy_RreXqnM5fqMR0hOLZPeXOvYoju'
DATA_PATH = Path('/content/cleveland.csv')
gdown.download(id=FILE_ID, output=str(DATA_PATH), quiet=False)

FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = [c for c in FEATURES if c not in NUMERICAL_FEATURES]

df = pd.read_csv(DATA_PATH, header=None, names=FEATURES + ['target_original'], na_values=['?'])
df['target'] = (pd.to_numeric(df['target_original'], errors='coerce') > 0).astype(int)
for column in FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df[FEATURES + ['target']].drop_duplicates().reset_index(drop=True)

train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['target'], random_state=RANDOM_STATE)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
print('Train locked:', train_df.shape, 'Test locked:', test_df.shape)
display(train_df.head())

## 2. Sinh dữ liệu x5/x10 theo từng lớp

Fit một Gaussian Copula riêng cho `target=0` và `target=1` để giữ tỷ lệ lớp. Sau sampling, ép kiểu/ràng buộc miền. x5 nghĩa là tổng train sau cùng bằng 5 lần train gốc.

In [ ]:
CLINICAL_BOUNDS = {
    'age': (18, 100), 'trestbps': (60, 260), 'chol': (80, 800),
    'thalach': (40, 240), 'oldpeak': (0.0, 10.0),
}
INTEGER_NUMERICAL = ['age','trestbps','chol','thalach']

def make_metadata(frame):
    metadata = Metadata.detect_from_dataframe(data=frame, table_name='patients')
    for column in CATEGORICAL_FEATURES:
        metadata.update_column(column_name=column, sdtype='categorical')
    for column in NUMERICAL_FEATURES:
        metadata.update_column(column_name=column, sdtype='numerical')
    return metadata

def nearest_allowed(series, allowed):
    allowed = np.asarray(sorted(allowed), dtype=float)
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = ~np.isnan(values)
    values[valid] = allowed[np.abs(values[valid, None] - allowed[None, :]).argmin(axis=1)]
    return values

def enforce_domains(sample, reference):
    result = sample.copy()
    for column, (low, high) in CLINICAL_BOUNDS.items():
        result[column] = pd.to_numeric(result[column], errors='coerce').clip(low, high)
    result[INTEGER_NUMERICAL] = result[INTEGER_NUMERICAL].round()
    result['oldpeak'] = result['oldpeak'].round(1)
    for column in CATEGORICAL_FEATURES:
        allowed = reference[column].dropna().unique()
        result[column] = nearest_allowed(result[column], allowed)
    return result[FEATURES]

def generate_expanded_train(real_train, multiplier, seed):
    np.random.seed(seed)
    synthetic_parts = []
    for label, class_frame in real_train.groupby('target'):
        features_only = class_frame[FEATURES].reset_index(drop=True)
        metadata = make_metadata(features_only)
        synthesizer = GaussianCopulaSynthesizer(
            metadata, enforce_min_max_values=True, enforce_rounding=True)
        synthesizer.fit(features_only)
        n_new = len(class_frame) * (multiplier - 1)
        sampled = synthesizer.sample(num_rows=n_new)
        sampled = enforce_domains(sampled, features_only)
        sampled['target'] = int(label)
        sampled['data_origin'] = 'synthetic'
        synthetic_parts.append(sampled)
    real_part = real_train.copy()
    real_part['data_origin'] = 'real'
    expanded = pd.concat([real_part, *synthetic_parts], ignore_index=True)
    return expanded.sample(frac=1, random_state=seed).reset_index(drop=True)

train_x5 = generate_expanded_train(train_df, multiplier=5, seed=RANDOM_STATE)
train_x10 = generate_expanded_train(train_df, multiplier=10, seed=RANDOM_STATE + 1)
print('Original:', train_df.shape, 'x5:', train_x5.shape, 'x10:', train_x10.shape)
display(pd.DataFrame({
    'original': train_df['target'].value_counts(normalize=True),
    'x5': train_x5['target'].value_counts(normalize=True),
    'x10': train_x10['target'].value_counts(normalize=True)}).round(4))

## 3. Quality gates cho synthetic data

KS càng thấp càng giống ở biến số; Total Variation càng thấp càng giống ở categorical. Real-vs-synthetic AUC gần 0.5 là khó phân biệt, gần 1.0 là synthetic khác real rõ rệt. Các chỉ số này chỉ là diagnostics, không chứng minh tính đúng y khoa.

In [ ]:
def total_variation(real, synthetic):
    categories = sorted(set(real.dropna().unique()) | set(synthetic.dropna().unique()))
    p = real.value_counts(normalize=True).reindex(categories, fill_value=0)
    q = synthetic.value_counts(normalize=True).reindex(categories, fill_value=0)
    return 0.5 * np.abs(p - q).sum()

def quality_report(real, expanded, name):
    synthetic = expanded[expanded['data_origin'] == 'synthetic']
    rows = []
    for column in NUMERICAL_FEATURES:
        a, b = real[column].dropna(), synthetic[column].dropna()
        rows.append({'dataset':name, 'feature':column, 'type':'numeric',
                     'distance':ks_2samp(a, b).statistic})
    for column in CATEGORICAL_FEATURES:
        rows.append({'dataset':name, 'feature':column, 'type':'categorical',
                     'distance':total_variation(real[column], synthetic[column])})
    exact_matches = synthetic[FEATURES].merge(real[FEATURES].drop_duplicates(), how='inner').shape[0]
    return pd.DataFrame(rows), exact_matches / max(len(synthetic), 1)

quality_x5, duplicate_rate_x5 = quality_report(train_df, train_x5, 'x5')
quality_x10, duplicate_rate_x10 = quality_report(train_df, train_x10, 'x10')
quality = pd.concat([quality_x5, quality_x10], ignore_index=True)
display(quality.pivot(index='feature', columns='dataset', values='distance').round(4))
print('Exact-match rate x5:', round(duplicate_rate_x5, 4))
print('Exact-match rate x10:', round(duplicate_rate_x10, 4))

plt.figure(figsize=(14, 5))
sns.boxplot(data=quality, x='feature', y='distance', hue='dataset')
plt.xticks(rotation=45); plt.title('Synthetic distribution distances'); plt.show()

## 4. Hospital corruption: nhẹ, trung bình, nặng

Mô phỏng missing, outlier/đơn vị sai, làm tròn, categorical code không hợp lệ và hospital shift. Nhãn test không bị sửa.

In [ ]:
NOISE_LEVELS = {
    'clean':  {'missing':0.00, 'outlier':0.00, 'code':0.00, 'round':0.00, 'shift':0.00},
    'mild':   {'missing':0.03, 'outlier':0.01, 'code':0.005,'round':0.05, 'shift':0.25},
    'medium': {'missing':0.08, 'outlier':0.03, 'code':0.01, 'round':0.10, 'shift':0.50},
    'severe': {'missing':0.15, 'outlier':0.05, 'code':0.02, 'round':0.20, 'shift':1.00},
}

def corrupt_hospital_data(frame, level, seed):
    config = NOISE_LEVELS[level]
    result = frame.copy().reset_index(drop=True)
    local_rng = np.random.default_rng(seed)
    if level == 'clean': return result
    missing_mask = local_rng.random((len(result), len(FEATURES))) < config['missing']
    result.loc[:, FEATURES] = result[FEATURES].mask(missing_mask)
    for column in NUMERICAL_FEATURES:
        mask = local_rng.random(len(result)) < config['outlier']
        factors = local_rng.choice([0.1, 0.5, 1.5, 10.0], size=mask.sum())
        result.loc[mask, column] = result.loc[mask, column] * factors
    rounding = {'trestbps':5, 'chol':10, 'thalach':5, 'oldpeak':0.5}
    for column, step in rounding.items():
        mask = local_rng.random(len(result)) < config['round']
        result.loc[mask, column] = (result.loc[mask, column] / step).round() * step
    for column in CATEGORICAL_FEATURES:
        mask = local_rng.random(len(result)) < config['code']
        result.loc[mask, column] = 99
    shifted = local_rng.random(len(result)) < config['shift']
    result.loc[shifted, 'age'] = result.loc[shifted, 'age'] + 3
    result.loc[shifted, 'trestbps'] = result.loc[shifted, 'trestbps'] + 5
    return result

test_sets = {name: corrupt_hospital_data(test_df, name, RANDOM_STATE+i)
             for i, name in enumerate(NOISE_LEVELS)}
noise_summary = []
for name, data in test_sets.items():
    noise_summary.append({'test':name, 'rows':len(data),
                          'missing_cells':int(data[FEATURES].isna().sum().sum()),
                          'invalid_codes':int((data[CATEGORICAL_FEATURES] == 99).sum().sum())})
display(pd.DataFrame(noise_summary))

## 5. Train lại model và đo robustness

So sánh train gốc, x5, x10, x5 bị noise trung bình và x10 bị noise trung bình. Preprocessing luôn fit trên từng training set.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                        ('scaler', MinMaxScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num', numeric, NUMERICAL_FEATURES),
                              ('cat', categorical, CATEGORICAL_FEATURES)])

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                             random_state=RANDOM_STATE, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2,
                                         random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                             subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
                             random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=300, num_leaves=15, learning_rate=0.03,
                               random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1),
}

train_sets = {
    'original': train_df.copy(),
    'x5_clean': train_x5.drop(columns='data_origin'),
    'x10_clean': train_x10.drop(columns='data_origin'),
    'x5_dirty': corrupt_hospital_data(train_x5.drop(columns='data_origin'), 'medium', 501),
    'x10_dirty': corrupt_hospital_data(train_x10.drop(columns='data_origin'), 'medium', 502),
}

records = []
for train_name, training in train_sets.items():
    for model_name, classifier in MODELS.items():
        pipeline = Pipeline([('preprocessor', make_preprocessor()), ('classifier', classifier)])
        started = time.perf_counter()
        pipeline.fit(training[FEATURES], training['target'])
        fit_seconds = time.perf_counter() - started
        for test_name, testing in test_sets.items():
            probability = pipeline.predict_proba(testing[FEATURES])[:, 1]
            prediction = (probability >= 0.5).astype(int)
            tn, fp, fn, tp = confusion_matrix(testing['target'], prediction, labels=[0,1]).ravel()
            records.append({
                'train_set':train_name, 'model':model_name, 'test_set':test_name,
                'accuracy':accuracy_score(testing['target'], prediction),
                'precision':precision_score(testing['target'], prediction, zero_division=0),
                'recall':recall_score(testing['target'], prediction, zero_division=0),
                'specificity':tn/(tn+fp) if tn+fp else np.nan,
                'false_negative_rate':fn/(fn+tp) if fn+tp else np.nan,
                'f1':f1_score(testing['target'], prediction, zero_division=0),
                'roc_auc':roc_auc_score(testing['target'], probability),
                'brier':brier_score_loss(testing['target'], probability),
                'fit_seconds':fit_seconds})
results = pd.DataFrame(records)
display(results.sort_values(['test_set','roc_auc'], ascending=[True,False]).round(4))

In [ ]:
clean = results[results['test_set']=='clean'][['train_set','model','roc_auc','recall','brier']].rename(
    columns={'roc_auc':'clean_auc','recall':'clean_recall','brier':'clean_brier'})
severe = results[results['test_set']=='severe'][['train_set','model','roc_auc','recall','brier']].rename(
    columns={'roc_auc':'severe_auc','recall':'severe_recall','brier':'severe_brier'})
robustness = clean.merge(severe, on=['train_set','model'])
robustness['auc_drop'] = robustness['clean_auc'] - robustness['severe_auc']
robustness['recall_drop'] = robustness['clean_recall'] - robustness['severe_recall']
robustness = robustness.sort_values(['severe_auc','auc_drop','severe_recall'],
                                    ascending=[False,True,False]).reset_index(drop=True)
display(robustness.round(4))

plt.figure(figsize=(13,6))
plot_df = results[results['train_set'].isin(['original','x5_clean','x10_clean'])]
sns.barplot(data=plot_df, x='model', y='roc_auc', hue='test_set', errorbar=None)
plt.xticks(rotation=25); plt.ylim(0.5,1.0); plt.title('ROC AUC under hospital corruption'); plt.show()

print('ROBUSTNESS LEADER')
display(robustness.head(1))

## 6. Quy tắc kết luận

Không chọn model chỉ vì x10 cho điểm cao nhất. Chỉ giữ synthetic khi: (1) quality distances chấp nhận được; (2) exact-match thấp; (3) clean test không giảm đáng kể; (4) severe-test AUC/Recall tốt hơn ổn định; và (5) Brier score không xấu đi. Mọi kết luận vẫn là exploratory vì test chỉ khoảng 61 bệnh nhân. Notebook không xuất model hay dữ liệu.